In [0]:
# Definimos el CATALOG aquí.

CATALOG = "data-engineer-associate"

In [0]:
from pyspark.sql.functions import col, trim

# ------------------------------------------------------------------
# STORES (silver)
# ------------------------------------------------------------------

# Leo la tabla bronze (todo STRING, sin transformar) como punto de partida.
stores_bronze_df = spark.table(f"`{CATALOG}`.`02_bronze`.stores_bronze")

stores_silver_df = (
    stores_bronze_df
    # cast(): aquí es donde SÍ convertimos tipos -- este es el paso de
    # "standardizing data types" que corresponde a la capa silver.
    .withColumn("store_id", col("store_id").cast("INT"))
    # trim(): quita espacios en blanco accidentales al inicio/final
    .withColumn("region", trim(col("region")))
    .withColumn("country", trim(col("country")))
)

# writeStream no aplica aquí -- esto es una tabla batch normal (no viene
# de un stream), así que uso .write en vez de .writeStream.
(
    stores_silver_df.write
    .format("delta")
    .mode("overwrite")   # cada corrida reemplaza la tabla completa
                          # (tiene sentido: son solo 20 filas, dimensión pequeña)
    .saveAsTable(f"`{CATALOG}`.`03_silver`.stores")
)

# ------------------------------------------------------------------
# PRODUCTS (silver)
# ------------------------------------------------------------------

products_bronze_df = spark.table(f"`{CATALOG}`.`02_bronze`.products_bronze")

products_silver_df = (
    products_bronze_df
    .withColumn("product_id", col("product_id").cast("INT"))
    .withColumn("name", trim(col("name")))
    .withColumn("category", trim(col("category")))
    .withColumn("price", col("price").cast("DOUBLE"))
)

(
    products_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{CATALOG}`.`03_silver`.products")
)

# ------------------------------------------------------------------
# Verificación + chequeo de calidad de datos básico
# ------------------------------------------------------------------

print("stores:", spark.table(f"`{CATALOG}`.`03_silver`.stores").count(), "filas")
print("products:", spark.table(f"`{CATALOG}`.`03_silver`.products").count(), "filas")

# Verifico que el casteo no haya generado NULLs inesperados
# (si algún store_id no era numérico, cast() lo hubiera vuelto NULL en vez
# de fallar -- por eso conviene chequear esto explícitamente)
nulls_check = spark.sql(f"""
    SELECT COUNT(*) as nulls_en_store_id
    FROM `{CATALOG}`.`03_silver`.stores
    WHERE store_id IS NULL
""")
display(nulls_check)

In [0]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import ArrayType, StructType, StructField, IntegerType, DoubleType

# ------------------------------------------------------------------
# 1. Defino el esquema esperado del array "items"
# ------------------------------------------------------------------
# from_json() necesita saber explícitamente qué estructura esperar
# dentro del string -- no lo infiere solo.
items_schema = ArrayType(
    StructType([
        StructField("product_id", IntegerType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DoubleType()),
    ])
)

# ------------------------------------------------------------------
# 2. Leo orders_bronze
# ------------------------------------------------------------------
orders_bronze_df = spark.table(f"`{CATALOG}`.`02_bronze`.orders_bronze")

# ------------------------------------------------------------------
# 3. from_json(): convierto el STRING de "items" en un array real,
#    tipado según el esquema de arriba
# ------------------------------------------------------------------
orders_with_array_df = orders_bronze_df.withColumn(
    "items_parsed",
    from_json(col("items"), items_schema)
)

# ------------------------------------------------------------------
# 4. explode(): "aplano" el array -- cada elemento del array se
#    convierte en su PROPIA FILA (esto es lo que crea order_items)
# ------------------------------------------------------------------
order_items_df = (
    orders_with_array_df
    .select(
        col("order_id").cast("INT"),
        explode(col("items_parsed")).alias("item")   # 1 fila por producto
    )
    # el struct "item" tiene sub-campos -- los saco a columnas normales
    .select(
        col("order_id"),
        col("item.product_id").alias("product_id"),
        col("item.quantity").alias("quantity"),
        col("item.unit_price").alias("unit_price"),
    )
)

(
    order_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{CATALOG}`.`03_silver`.order_items")
)

# ------------------------------------------------------------------
# Verificación
# ------------------------------------------------------------------
print("order_items:", spark.table(f"`{CATALOG}`.`03_silver`.order_items").count(), "filas")
display(spark.table(f"`{CATALOG}`.`03_silver`.order_items").limit(10))

In [0]:
from pyspark.sql.functions import col, to_timestamp

# ------------------------------------------------------------------
# ORDERS (silver) -- encabezado, SIN la columna items
# (items ya se extrajo aparte en order_items)
# ------------------------------------------------------------------

orders_bronze_df = spark.table(f"`{CATALOG}`.`02_bronze`.orders_bronze")

orders_silver_df = (
    orders_bronze_df
    .select(
        col("order_id").cast("INT"),
        col("customer_id").cast("INT"),
        col("store_id").cast("INT"),
        to_timestamp(col("order_ts")).alias("order_ts"),  # string -> TIMESTAMP real
        col("channel"),
    )
    # Nota: "items" NO se incluye aquí a propósito -- ya vive,
    # aplanado, en la tabla order_items que creamos antes.
)

(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{CATALOG}`.`03_silver`.orders")
)

print("orders:", spark.table(f"`{CATALOG}`.`03_silver`.orders").count(), "filas")
display(spark.table(f"`{CATALOG}`.`03_silver`.orders").limit(10))

In [0]:
from pyspark.sql.functions import sum as spark_sum, first, col

order_items_dedup_df = (
    spark.table(f"`{CATALOG}`.`03_silver`.order_items")
    .groupBy("order_id", "product_id")
    .agg(
        spark_sum("quantity").alias("quantity"),
        first("unit_price").alias("unit_price")
    )
    # FIX: sum() devuelve BIGINT por defecto -- lo regreso a INT
    # explícitamente para que coincida con el esquema ya declarado
    # en la tabla order_items.
    .withColumn("quantity", col("quantity").cast("INT"))
)

(
    order_items_dedup_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"`{CATALOG}`.`03_silver`.order_items")
)

print("order_items (después de consolidar):", spark.table(f"`{CATALOG}`.`03_silver`.order_items").count(), "filas")

In [0]:
from pyspark.sql.functions import approx_count_distinct, broadcast, to_date, date_format, col

# ============================================================
# 1. DEDUPLICACIÓN -- ¿hay order_id repetidos donde no deberían?
# ============================================================
print("--- Duplicados en orders (por order_id) ---")
dup_orders = (
    spark.table(f"`{CATALOG}`.`03_silver`.orders")
    .groupBy("order_id")
    .count()
    .filter("count > 1")
)
print(f"Filas duplicadas encontradas: {dup_orders.count()}")  # debería ser 0
dup_orders.show(5)

print("--- Duplicados en order_items (por order_id + product_id) ---")
dup_items = (
    spark.table(f"`{CATALOG}`.`03_silver`.order_items")
    .groupBy("order_id", "product_id")
    .count()
    .filter("count > 1")
)
print(f"Filas duplicadas encontradas: {dup_items.count()}")  # puede NO ser 0:
# si un mismo producto aparece 2 veces en la misma orden (posible con
# nuestros datos sintéticos), no es un error, es un caso real a considerar
# en gold (¿sumar cantidades o dejar filas separadas?)


# ============================================================
# 2. INTEGRIDAD REFERENCIAL -- ¿hechos "huérfanos" sin dimensión?
# ============================================================
print("\n--- order_items con product_id que NO existe en products ---")
huerfanos_productos = (
    spark.table(f"`{CATALOG}`.`03_silver`.order_items").alias("oi")
    .join(
        spark.table(f"`{CATALOG}`.`03_silver`.products").alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left_anti"   # left_anti: me quedo SOLO con lo que NO matcheó
    )
)
print(f"Huérfanos encontrados: {huerfanos_productos.count()}")  # debería ser 0

print("\n--- orders con store_id que NO existe en stores ---")
huerfanos_stores = (
    spark.table(f"`{CATALOG}`.`03_silver`.orders").alias("o")
    .join(
        spark.table(f"`{CATALOG}`.`03_silver`.stores").alias("s"),
        col("o.store_id") == col("s.store_id"),
        "left_anti"
    )
)
print(f"Huérfanos encontrados: {huerfanos_stores.count()}")  # debería ser 0


# ============================================================
# 3. AGREGACIONES / RESUMEN ESTADÍSTICO
# ============================================================
print("\n--- Resumen estadístico de order_items ---")
spark.table(f"`{CATALOG}`.`03_silver`.order_items").summary().show()
# summary() da count, mean, stddev, min, percentiles, max por columna numérica
# -- útil para detectar outliers (ej. un unit_price negativo o absurdamente alto)

print("\n--- Conteo aproximado de clientes únicos que compraron ---")
spark.table(f"`{CATALOG}`.`03_silver`.orders") \
    .select(approx_count_distinct("customer_id").alias("clientes_unicos_aprox")) \
    .show()
# approx_count_distinct es más rápido que countDistinct() en datasets grandes
# -- usa el algoritmo HyperLogLog, da un estimado, no el valor exacto


# ============================================================
# 4. MANIPULACIÓN DE COLUMNAS -- dividir, renombrar
# ============================================================
print("\n--- Ejemplo: dividir order_ts en fecha/hora, renombrar channel ---")
orders_demo = (
    spark.table(f"`{CATALOG}`.`03_silver`.orders")
    .withColumn("order_date", to_date("order_ts"))          # DIVIDIR: fecha
    .withColumn("order_time", date_format("order_ts", "HH:mm:ss"))  # DIVIDIR: hora
    .withColumnRenamed("channel", "sales_channel")           # RENOMBRAR
)
orders_demo.select("order_id", "order_ts", "order_date", "order_time", "sales_channel").show(5)
# Nota: esto es solo un EJEMPLO para practicar la sintaxis -- no sobrescribe
# la tabla orders real. Si quieres quedarte con estas columnas, habría que
# guardar orders_demo con .write.saveAsTable(...) reemplazando la tabla.


# ============================================================
# 5. BROADCAST JOIN explícito
# ============================================================
print("\n--- Plan de ejecución de un broadcast join forzado ---")
joined = (
    spark.table(f"`{CATALOG}`.`03_silver`.order_items")
    .join(
        broadcast(spark.table(f"`{CATALOG}`.`03_silver`.products")),  # products es chica (200 filas)
        "product_id"
    )
)
joined.explain()
# Busca "BroadcastHashJoin" en el plan impreso -- confirma que Spark
# distribuyó una copia completa de products a cada executor, en vez
# de hacer shuffle de las 1.2M filas de order_items